# Bio-Med-Llama-3-8B evaluation notebook — 12 combinations

This notebook evaluates **aaditya/OpenBioLLM-Llama3-8B** with an optional **LoRA verifier** over **3 datasets** and **5 experiment methods** (3 × 5 = **15 combinations**).

**Datasets:**

- `mimic`
- `openi`
- `combined`

**Methods:**

- `faiss_majority_vote` — **pure FAISS retrieval baseline, no LLM**: for each of the 13 CheXpert labels, takes the majority vote across the top-10 retrieved neighbor metadata labels; no model inference
- `base_noverify_noconfess` — single-pass baseline, no LoRA, no confession
- `base_confess` — base model with full 4-pass confession loop, **no LoRA verifier**
- `verify_noconfess` — LoRA verifier active, no confession
- `verify_confess` — LoRA verifier + full confession loop

It supports:

- **text-only** RAG: query image → BioMedCLIP FAISS retrieval → top-10 text reports → text-only LLM generation
- LoRA on / off adapter switching on the same wrapped model object
- verifier-guided confession logic with majority / minority retrieval reasoning
- final metrics: label accuracy, hallucination / omission, text similarity, RadGraph F1
- automatic 15-combination comparison CSV generation
- `faiss_majority_vote` pure retrieval baseline (label majority vote, no LLM inference)

> **Key design note:** OpenBioLLM-Llama3-8B is a **text-only** model.
> The query image is used solely for BioMedCLIP FAISS retrieval of the top-k
> similar text reports, which are then fed as text context to the LLM.
>
> **Chat-template note:** `aaditya/OpenBioLLM-Llama3-8B` does not ship
> `tokenizer.chat_template` in all HF versions. `USE_CHAT_TEMPLATE = False`
> is set in cell 1 so that `render_chat_text()` uses a safe plain-prompt
> fallback instead of `apply_chat_template()` (which would raise a
> `ValueError` if the template is absent).


In [1]:
# ============================================================
# 0) Configuration
# ============================================================
from __future__ import annotations

from pathlib import Path
import os

# -------- Bio-Med-Llama-3-8B model + LoRA verifier --------
BASE_MODEL_ID = "aaditya/OpenBioLLM-Llama3-8B"
LORA_PATH     = Path("/data/liangz2/openi/biomed_llama/lora")
JSONL_PATH    = Path("/data/liangz2/openi/harmony_set/openi_cxr_harmony_rl.jsonl")
HF_TOKEN_PATH = Path("/data/liangz2/openi/hf_token.txt")

# -------- Dataset roots — all three active for 12-combo run --------
DATASET_ROOTS = {
    "mimic"   : Path("/data/liangz2/openi/faiss_val_mimic_biomedclip"),
    "openi"   : Path("/data/liangz2/openi/faiss_val_openi_biomedclip"),
    "combined": Path("/data/liangz2/openi/faiss_val_combined_biomedclip"),
}

# -------- 5 experiment methods (3 datasets × 5 = 15 combinations) --------
EXPERIMENTS = [
    {
        # Pure FAISS majority-vote baseline — no LLM call at all.
        # Predicts each of the 13 CheXpert labels by majority vote of the
        # top-10 retrieved neighbor metadata labels. Fastest method; serves
        # as a retrieval-only reference baseline for the LLM-based methods.
        "name": "faiss_majority_vote",
        "USE_LORA": False,
        "USE_CONFESSION": False,
        "USE_FAISS_MAJORITY": True,
        "subdir": "faiss_majority_vote",
    },
    # {
    #     "name": "base_noverify_noconfess",
    #     "USE_LORA": False,
    #     "USE_CONFESSION": False,
    #     "USE_FAISS_MAJORITY": False,
    #     "subdir": "biomed_llama_Noverify_Noconfess",
    # },
    # {
    #     # base model with confession loop — no LoRA verifier
    #     # Runs all 4 passes unconditionally (no early-exit gating).
    #     "name": "base_confess",
    #     "USE_LORA": False,
    #     "USE_CONFESSION": True,
    #     "USE_FAISS_MAJORITY": False,
    #     "subdir": "biomed_llama_base_confess",
    # },
    # {
    #     "name": "verify_noconfess",
    #     "USE_LORA": True,
    #     "USE_CONFESSION": False,
    #     "USE_FAISS_MAJORITY": False,
    #     "subdir": "biomed_llama_verify_Noconfess",
    # },
    # {
    #     "name": "verify_confess",
    #     "USE_LORA": True,
    #     "USE_CONFESSION": True,
    #     "USE_FAISS_MAJORITY": False,
    #     "subdir": "biomed_llama_verify_confess",
    # },
]

# -------- Generation / retrieval --------
TOP_K          = 10
N_ITEMS        = None          # set int for debugging
PROGRESS_EVERY = 25
STOP_IF_UNCHANGED = False

# -------- Chat template flag --------
# aaditya/OpenBioLLM-Llama3-8B does NOT ship tokenizer.chat_template in all
# HF versions.  Setting USE_CHAT_TEMPLATE=False makes render_chat_text() use
# a safe plain-prompt fallback (no apply_chat_template call, no ValueError).
USE_CHAT_TEMPLATE = False

# -------- Generation controls --------
MAX_INPUT_LEN  = 4096
MAX_NEW_TOKENS = 256
DO_SAMPLE      = False
TEMPERATURE    = 0.2
TOP_P          = 0.95

# -------- Output --------
COMPARISON_CSV = Path("/data/liangz2/openi/biomed_llama_all_dataset_15combo_comparison.csv")

print("BASE_MODEL_ID:", BASE_MODEL_ID)
print("LORA_PATH    :", LORA_PATH)
print("JSONL_PATH   :", JSONL_PATH)
print("COMPARISON_CSV:", COMPARISON_CSV)
print(f"Experiments  : {len(EXPERIMENTS)} methods × {len(DATASET_ROOTS)} datasets = "
      f"{len(EXPERIMENTS)*len(DATASET_ROOTS)} combinations")
print("  incl. faiss_majority_vote: pure FAISS majority-vote, no LLM")
for k, v in DATASET_ROOTS.items():
    print(k, "->", v)


BASE_MODEL_ID: aaditya/OpenBioLLM-Llama3-8B
LORA_PATH    : /data/liangz2/openi/biomed_llama/lora
JSONL_PATH   : /data/liangz2/openi/harmony_set/openi_cxr_harmony_rl.jsonl
COMPARISON_CSV: /data/liangz2/openi/biomed_llama_all_dataset_15combo_comparison.csv
Experiments  : 1 methods × 3 datasets = 3 combinations
  incl. faiss_majority_vote: pure FAISS majority-vote, no LLM
mimic -> /data/liangz2/openi/faiss_val_mimic_biomedclip
openi -> /data/liangz2/openi/faiss_val_openi_biomedclip
combined -> /data/liangz2/openi/faiss_val_combined_biomedclip


In [2]:
# ============================================================
# 1) Login + imports
# ============================================================
from huggingface_hub import login

if HF_TOKEN_PATH.exists():
    with open(HF_TOKEN_PATH, "r", encoding="utf-8") as f:
        hf_token = f.readline().strip()
    if hf_token:
        login(token=hf_token)
        print("✅ Hugging Face login successful.")
    else:
        print("⚠️ HF token file is empty.")
else:
    print(f"⚠️ HF token file not found: {HF_TOKEN_PATH}")

import csv
import json
import math
import re
import time
from collections import Counter
from typing import Any, Dict, List, Optional, Tuple

import faiss
import numpy as np
import open_clip
import torch
from PIL import Image
from peft import PeftModel

from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForCausalLM,
)

# RadGraph compatibility patch
import transformers
from torch.optim import AdamW as TorchAdamW
if not hasattr(transformers, "AdamW"):
    transformers.AdamW = TorchAdamW

print("Imports ready.")


✅ Hugging Face login successful.
Imports ready.


In [3]:
# ============================================================
# 2) Load Bio-Med-Llama-3-8B backbone + LoRA wrapper (text-only)
# ============================================================
# Try loading the tokenizer from the LoRA directory first — the fine-tuning
# run may have saved a complete tokenizer_config.json there.
# Fall back to the base model tokenizer if no tokenizer files exist in LORA_PATH.
try:
    tokenizer = AutoTokenizer.from_pretrained(
        str(LORA_PATH),
        use_fast=True,
        trust_remote_code=True,
        local_files_only=True,
    )
    print("Loaded tokenizer from LoRA directory:", LORA_PATH)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    print("Loaded tokenizer from base model:", BASE_MODEL_ID)

if getattr(tokenizer, "pad_token", None) is None:
    tokenizer.pad_token = tokenizer.eos_token

config = AutoConfig.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

backbone = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=dtype,
    device_map="auto",
)
backbone.eval()

verifier_base = PeftModel.from_pretrained(backbone, str(LORA_PATH))
verifier_base.eval()

print("✅ Bio-Med-Llama-3-8B backbone + LoRA wrapper loaded")
print("Backbone type:", type(backbone))
print("Verifier type:", type(verifier_base))
print("Device:", next(verifier_base.parameters()).device)
print("Tokenizer has chat_template:", bool(getattr(tokenizer, "chat_template", None)))


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded tokenizer from LoRA directory: /data/liangz2/openi/biomed_llama/lora


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Bio-Med-Llama-3-8B backbone + LoRA wrapper loaded
Backbone type: <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
Verifier type: <class 'peft.peft_model.PeftModelForCausalLM'>
Device: cuda:0
Tokenizer has chat_template: False


In [4]:
# ============================================================
# 3) Bio-Med-Llama-3-8B prompt helpers + robust LoRA toggling + generation / verification
#    NOTE: This is a TEXT-ONLY model. No image inputs are passed to the LLM.
#    The query image is used only for BioMedCLIP FAISS retrieval.
# ============================================================
from contextlib import nullcontext

BIOLLAMA_SYSTEM = (
    "You are a radiology assistant for chest X-ray interpretation. "
    "Use the retrieved reference evidence to write a concise final chest X-ray report. "
    "Return only report text with FINDINGS and IMPRESSION. "
    "Do not answer TRUE or FALSE. Do not output JSON, bullets, markdown, or extra commentary."
)

VERIFY_SYSTEM = (
    "You are a radiology verification assistant. "
    "Decide whether the candidate report is supported by the retrieved evidence. "
    "Answer exactly TRUE or FALSE."
)

def _messages_to_plain_prompt(messages: List[Dict[str, str]]) -> str:
    """
    Fallback for models without tokenizer.chat_template.

    aaditya/OpenBioLLM-Llama3-8B does not include chat_template in all HF
    tokenizer_config.json versions.  Calling apply_chat_template() on such a
    tokenizer raises:
        ValueError: Cannot use chat template functions because
                    tokenizer.chat_template is not set ...
    This helper safely concatenates system + user message content into a flat
    string that any causal-LM tokenizer can consume without special tokens.
    """
    parts = []
    for m in messages:
        role    = (m.get("role") or "user").strip()
        content = (m.get("content") or "").strip()
        if not content:
            continue
        if role in ("system", "user"):
            parts.append(content)
        # assistant turns are not expected in our generation pipeline
    return "\n\n".join(parts)


def render_chat_text(tokenizer, messages: List[Dict[str, str]], add_generation_prompt: bool) -> str:
    """
    Safely render chat messages to a prompt string.

    - If USE_CHAT_TEMPLATE is True AND tokenizer.chat_template is set
      → apply_chat_template (preferred for models with a saved template)
    - Otherwise
      → plain-prompt fallback via _messages_to_plain_prompt()
      (safe for Bio-Med-Llama and any model without tokenizer.chat_template)
    """
    can_chat = (
        USE_CHAT_TEMPLATE
        and bool(getattr(tokenizer, "chat_template", None))
    )
    if can_chat:
        return tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=add_generation_prompt,
            tokenize=False,
        )
    # Plain-prompt fallback — no apply_chat_template call, no ValueError
    return _messages_to_plain_prompt(messages)

def model_device(model) -> torch.device:
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# Utilities: robust adapter toggling
# ----------------------------
def adapter_disabled_ctx(model: PeftModel):
    """
    Prefer PEFT's context-manager-based disable_adapter().
    Fall back to a no-op if unavailable.
    """
    try:
        ctx = model.disable_adapter()
        return ctx if ctx is not None else nullcontext()
    except Exception:
        return nullcontext()

def force_default_adapter(model: PeftModel):
    """
    Best-effort re-enable of the default adapter after generation.
    """
    try:
        model.enable_adapter()
        return
    except Exception:
        pass

    try:
        active = getattr(model, "active_adapter", None)
        if active:
            model.set_adapter(active)
        else:
            model.set_adapter("default")
    except Exception:
        pass

def lora_off(model: PeftModel):
    return adapter_disabled_ctx(model)

def lora_on(model: PeftModel):
    force_default_adapter(model)

def _prepare_text_inputs(tokenizer, prompt_text: str, device: torch.device):
    """Tokenise text-only input for Bio-Med-Llama-3-8B (no image)."""
    enc = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN,
    )
    model_inputs = {k: v.to(device) for k, v in enc.items()}
    return model_inputs

def clean_generated_report_text(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(r"\s+", " ", text).strip()
    if text.upper() in {"TRUE", "FALSE"}:
        return text.upper()
    text = re.sub(r"^(assistant\s*:?)+", "", text, flags=re.I).strip()
    return text

# ----------------------------
# Generation pass (LoRA OFF) — TEXT-ONLY
# ----------------------------
@torch.inference_mode()
def generate_report_base(
    model: PeftModel,
    tokenizer,
    prompt_messages: List[Dict[str, str]],
    max_new_tokens: int = MAX_NEW_TOKENS,
    do_sample: bool = DO_SAMPLE,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
) -> str:
    dev = model_device(model)
    prompt_text = render_chat_text(tokenizer, prompt_messages, add_generation_prompt=True)
    model_inputs = _prepare_text_inputs(tokenizer, prompt_text, dev)

    # Always disable LoRA for generation — verifier adapter should NOT influence
    # report style. For base_confess mode the adapter is simply not loaded.
    with adapter_disabled_ctx(model):
        out_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    force_default_adapter(model)

    gen_ids = out_ids[0][model_inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return clean_generated_report_text(text)

# ----------------------------
# Verification pass (LoRA ON, text-only)
# Only called when use_lora=True.
# ----------------------------
@torch.inference_mode()
def verify_true_false_lora(
    model: PeftModel,
    tokenizer,
    verify_prompt: str,
) -> dict:
    force_default_adapter(model)

    messages = [
        {"role": "system", "content": VERIFY_SYSTEM},
        {"role": "user", "content": verify_prompt},
    ]
    prompt_text = render_chat_text(tokenizer, messages, add_generation_prompt=True)

    dev = model_device(model)
    model_inputs = _prepare_text_inputs(tokenizer, prompt_text, dev)

    logits = model(**model_inputs).logits
    next_logits = logits[:, -1, :]

    true_ids  = tokenizer.encode("TRUE",  add_special_tokens=False)
    false_ids = tokenizer.encode("FALSE", add_special_tokens=False)

    if len(true_ids) == 1 and len(false_ids) == 1:
        tid, fid = true_ids[0], false_ids[0]
        two   = torch.stack([next_logits[0, tid], next_logits[0, fid]], dim=0)
        probs = torch.softmax(two, dim=0)
        p_true  = float(probs[0].item())
        p_false = float(probs[1].item())
        pred    = "TRUE" if p_true >= p_false else "FALSE"
        return {"pred": pred, "p_true": p_true, "p_false": p_false}

    gen = model.generate(
        **model_inputs,
        max_new_tokens=2,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    ans    = tokenizer.decode(gen[0][model_inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    ans_up = ans.upper()
    if "TRUE"  in ans_up and "FALSE" not in ans_up:
        return {"pred": "TRUE",    "p_true": 1.0, "p_false": 0.0}
    if "FALSE" in ans_up and "TRUE"  not in ans_up:
        return {"pred": "FALSE",   "p_true": 0.0, "p_false": 1.0}
    return     {"pred": "UNKNOWN", "p_true": float("nan"), "p_false": float("nan")}


In [5]:
# ============================================================
# 4) Load BioMedCLIP for FAISS query embeddings
#    (Image encoder used ONLY for retrieval, NOT passed to Bio-Med-Llama-3-8B)
# ============================================================
BIOCLIP_MODEL_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"

biomedclip_model, biomedclip_preprocess = open_clip.create_model_from_pretrained(BIOCLIP_MODEL_ID)
biomedclip_tokenizer = open_clip.get_tokenizer(BIOCLIP_MODEL_ID)
biomedclip_model = biomedclip_model.to(model_device(verifier_base)).eval()

@torch.inference_mode()
def biomedclip_image_embedding(image_path: str, l2_normalize: bool = True) -> np.ndarray:
    img  = Image.open(image_path).convert("RGB")
    x    = biomedclip_preprocess(img).unsqueeze(0).to(model_device(verifier_base))
    feat = biomedclip_model.encode_image(x)
    feat = feat.float()
    if l2_normalize:
        feat = feat / feat.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return feat[0].detach().cpu().numpy().astype(np.float32)


In [6]:
# ============================================================
# 5) FAISS helpers (gallery root == test root)
# ============================================================

def load_faiss_bundle(faiss_dir: str | Path) -> Tuple[faiss.Index, List[Dict[str, Any]]]:
    faiss_dir = Path(faiss_dir)
    index_path = faiss_dir / "faiss_image.index"
    meta_path = faiss_dir / "metadata.jsonl"

    if not index_path.exists():
        raise FileNotFoundError(f"Missing: {index_path}")
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing: {meta_path}")

    index = faiss.read_index(str(index_path))
    meta = []
    with open(meta_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                meta.append(json.loads(line))
    return index, meta


def load_dataset_bundles(dataset_root: str | Path) -> Tuple[faiss.Index, List[Dict[str, Any]], faiss.Index, List[Dict[str, Any]]]:
    dataset_root = Path(dataset_root)
    test_index, test_meta = load_faiss_bundle(dataset_root)
    gal_index, gal_meta = load_faiss_bundle(dataset_root)   # same path as requested
    print(f"✅ Loaded dataset bundles for {dataset_root.name}")
    print("test size:", len(test_meta), "gallery size:", len(gal_meta))
    return test_index, test_meta, gal_index, gal_meta


def query_vec_from_test_id(test_index: faiss.Index, test_id: int) -> np.ndarray:
    vec = test_index.reconstruct(int(test_id))
    vec = np.asarray(vec, dtype=np.float32)
    if vec.ndim == 1:
        vec = vec[None, :]
    return vec


def retrieve_gallery_neighbors(gallery_index: faiss.Index, gallery_meta: List[Dict[str, Any]], qvec: np.ndarray, k: int = 10) -> List[Dict[str, Any]]:
    D, I = gallery_index.search(qvec.astype(np.float32), int(k))
    neighbors: List[Dict[str, Any]] = []
    for rank, (idx, dist) in enumerate(zip(I[0].tolist(), D[0].tolist()), start=1):
        if idx < 0 or idx >= len(gallery_meta):
            continue
        row = dict(gallery_meta[idx])
        row["_rank"] = rank
        row["_score"] = float(dist)
        neighbors.append(row)
    return neighbors

In [7]:
# ============================================================
# 6) Evidence formatting + target report helpers
# ============================================================
NO_DETAIL = "No detailed information"

def _safe_str(x: Any) -> str:
    return (str(x) if x is not None else "").strip()

def _row_caption(row: Dict[str, Any]) -> str:
    for k in ["caption", "pred_caption", "report", "report_text"]:
        v = _safe_str(row.get(k, ""))
        if v:
            return v
    return NO_DETAIL

def _row_findings(row: Dict[str, Any]) -> str:
    for k in ["FINDINGS", "findings"]:
        v = _safe_str(row.get(k, ""))
        if v:
            return v
    return NO_DETAIL

def _row_impression(row: Dict[str, Any]) -> str:
    for k in ["IMPRESSION", "impression"]:
        v = _safe_str(row.get(k, ""))
        if v:
            return v
    return NO_DETAIL

def _row_normal(row: Dict[str, Any]) -> str:
    v = _safe_str(row.get("normal", "")).lower()
    return v if v in {"yes", "no"} else "unknown"

def _row_labels(row: Dict[str, Any]) -> List[str]:
    labs = row.get("labels", [])
    if isinstance(labs, list):
        return [str(x).strip().lower() for x in labs if str(x).strip()]
    return []

def build_reference_block(neighbors: List[Dict[str, Any]], k: int) -> str:
    top = neighbors[: max(1, min(int(k), len(neighbors)))]
    blocks = []
    for i, row in enumerate(top, start=1):
        blocks.append(
            f"Reference {i}:\n"
            f"normal: {_row_normal(row)}\n"
            f"labels: {_row_labels(row)}\n"
            f"caption: {_row_caption(row)}\n"
            f"FINDINGS: {_row_findings(row)}\n"
            f"IMPRESSION: {_row_impression(row)}"
        )
    return "\n\n".join(blocks)

def build_gt_report_text(row: Dict[str, Any]) -> str:
    findings = _row_findings(row)
    impression = _row_impression(row)
    caption = _row_caption(row)

    parts = []
    if findings and findings != NO_DETAIL:
        parts.append(f"FINDINGS: {findings}")
    if impression and impression != NO_DETAIL:
        parts.append(f"IMPRESSION: {impression}")
    if not parts and caption and caption != NO_DETAIL:
        parts.append(caption)
    return "\n".join(parts).strip()

In [8]:
# ============================================================
# 7) Prompt builders for initial pass + verifier-guided confession
#    NOTE: Text-only prompts — no image references in the LLM input.
#    "Query image" is used only for FAISS retrieval; the LLM sees only
#    the retrieved text reports as context.
#    These builders are shared by all 4 experiment methods.
# ============================================================

def split_majority_minority(neighbors: List[Dict[str, Any]], k: int = 10) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], str]:
    top = neighbors[: max(1, min(int(k), len(neighbors)))]
    yes_rows = [r for r in top if _row_normal(r) == "yes"]
    no_rows = [r for r in top if _row_normal(r) == "no"]

    if len(yes_rows) >= len(no_rows):
        majority_flag = "yes"
        majority_rows = yes_rows if yes_rows else top
        minority_rows = no_rows
    else:
        majority_flag = "no"
        majority_rows = no_rows if no_rows else top
        minority_rows = yes_rows

    return majority_rows, minority_rows, majority_flag

def build_reference_block_with_tag(neighbors: List[Dict[str, Any]], tag: str, k: Optional[int] = None) -> str:
    use_rows = neighbors if k is None else neighbors[: max(1, min(int(k), len(neighbors)))]
    blocks = []
    for i, row in enumerate(use_rows, start=1):
        blocks.append(
            f"{tag} Reference {i}:\n"
            f"normal: {_row_normal(row)}\n"
            f"labels: {_row_labels(row)}\n"
            f"caption: {_row_caption(row)}\n"
            f"FINDINGS: {_row_findings(row)}\n"
            f"IMPRESSION: {_row_impression(row)}"
        )
    return "\n\n".join(blocks)

def build_initial_messages(neighbors: List[Dict[str, Any]], k: int = 10) -> List[Dict[str, str]]:
    evidence = build_reference_block(neighbors, k=k)
    user = (
        "Use the retrieved reference cases below to write the final chest X-ray report for the query study.\n\n"
        "Return only the final report text with concise FINDINGS and IMPRESSION.\n\n"
        f"{evidence}"
    )
    return [
        {"role": "system", "content": BIOLLAMA_SYSTEM},
        {"role": "user", "content": user},
    ]

def build_majority_messages(majority_neighbors: List[Dict[str, Any]], prev_report: str, pass_id: int) -> List[Dict[str, str]]:
    evidence = build_reference_block_with_tag(majority_neighbors, tag="MAJORITY", k=len(majority_neighbors))
    user = (
        f"This is confession pass {pass_id} for the same query study.\n\n"
        "Revise the previous draft using ONLY the majority retrieved evidence below.\n"
        "Treat these majority references as the currently most reliable guidance.\n"
        "Return only the revised final report text.\n\n"
        f"Previous draft:\n{prev_report}\n\n"
        f"Majority retrieved evidence:\n{evidence}"
    )
    return [
        {"role": "system", "content": BIOLLAMA_SYSTEM},
        {"role": "user", "content": user},
    ]

def build_minority_false_messages(minority_neighbors: List[Dict[str, Any]], prev_report: str, pass_id: int) -> List[Dict[str, str]]:
    evidence = build_reference_block_with_tag(minority_neighbors, tag="MINORITY_FALSE", k=len(minority_neighbors)) if minority_neighbors else "No minority evidence."
    user = (
        f"This is confession pass {pass_id} for the same query study.\n\n"
        "The minority retrieved evidence below should be treated as likely FALSE statements or misleading evidence.\n"
        "Use them only to avoid copying their false claims into the report.\n"
        "Revise the previous draft accordingly and return only the revised final report text.\n\n"
        f"Previous draft:\n{prev_report}\n\n"
        f"Minority retrieved evidence annotated as FALSE:\n{evidence}"
    )
    return [
        {"role": "system", "content": BIOLLAMA_SYSTEM},
        {"role": "user", "content": user},
    ]

def build_mixed_final_messages(
    majority_neighbors: List[Dict[str, Any]],
    minority_neighbors: List[Dict[str, Any]],
    prev_report: str,
    pass_id: int,
) -> List[Dict[str, str]]:
    majority_text = build_reference_block_with_tag(majority_neighbors, tag="MAJORITY_TRUE", k=len(majority_neighbors))
    minority_text = build_reference_block_with_tag(minority_neighbors, tag="MINORITY_FALSE", k=len(minority_neighbors)) if minority_neighbors else "No minority evidence."
    user = (
        f"This is the final confession pass {pass_id} for the same query study.\n\n"
        "Use the MAJORITY_TRUE evidence as reliable support.\n"
        "Use the MINORITY_FALSE evidence as likely misleading or false statements to avoid.\n"
        "Revise the previous draft one final time and return only the final report text.\n\n"
        f"Previous draft:\n{prev_report}\n\n"
        f"MAJORITY_TRUE evidence:\n{majority_text}\n\n"
        f"MINORITY_FALSE evidence:\n{minority_text}"
    )
    return [
        {"role": "system", "content": BIOLLAMA_SYSTEM},
        {"role": "user", "content": user},
    ]

def build_verify_prompt(report_text: str, neighbors: List[Dict[str, Any]], mode: str = "majority") -> str:
    majority_neighbors, minority_neighbors, majority_flag = split_majority_minority(neighbors, k=len(neighbors))

    if mode == "majority":
        evidence = build_reference_block_with_tag(majority_neighbors, tag="MAJORITY", k=len(majority_neighbors))
        instruction = "Decide whether the draft report is supported by the majority retrieved evidence. Answer exactly TRUE or FALSE."
    elif mode == "minority_false":
        evidence = build_reference_block_with_tag(minority_neighbors, tag="MINORITY_FALSE", k=len(minority_neighbors)) if minority_neighbors else "No minority evidence."
        instruction = "Decide whether the draft report avoids the likely false minority evidence below. Answer exactly TRUE or FALSE."
    else:
        majority_text = build_reference_block_with_tag(majority_neighbors, tag="MAJORITY_TRUE", k=len(majority_neighbors))
        minority_text = build_reference_block_with_tag(minority_neighbors, tag="MINORITY_FALSE", k=len(minority_neighbors)) if minority_neighbors else "No minority evidence."
        evidence = f"{majority_text}\n\n{minority_text}"
        instruction = "Decide whether the draft report is supported by the majority evidence and avoids the minority false evidence. Answer exactly TRUE or FALSE."

    return (
        f"{instruction}\n\n"
        f"Draft report:\n{report_text}\n\n"
        f"Evidence:\n{evidence}"
    )

def normalize_report_text(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

In [9]:
# ============================================================
# 8) Bio-Med-Llama-3-8B RAG report generation with verifier-guided confession
#    TEXT-ONLY: image_path is used ONLY for BioMedCLIP FAISS retrieval;
#    the LLM receives only the retrieved text reports as context.
#
#    Behaviour by (use_lora, confession):
#      (False, False) -> 1 pass  [base_noverify_noconfess]
#      (False, True)  -> 4 passes, no verify gating  [base_confess] ← NEW
#      (True,  False) -> 1 pass + verify score stored  [verify_noconfess]
#      (True,  True)  -> up-to-4 passes with verify-gated early exit  [verify_confess]
# ============================================================

def rag_generate_biollama_report(
    *,
    model: PeftModel,
    tokenizer,
    neighbors: List[Dict[str, Any]],
    k: int = 10,
    use_lora: bool = True,
    confession: bool = False,
) -> Dict[str, Any]:
    history: List[Dict[str, Any]] = []
    majority_neighbors, minority_neighbors, majority_flag = split_majority_minority(neighbors, k=k)

    # Pass 1: initial generation using full top-k retrieval
    messages = build_initial_messages(neighbors, k=k)
    report_1 = generate_report_base(
        model=model,
        tokenizer=tokenizer,
        prompt_messages=messages,
    )

    verify_1 = None
    if use_lora:
        verify_1 = verify_true_false_lora(
            model=model,
            tokenizer=tokenizer,
            verify_prompt=build_verify_prompt(report_1, neighbors, mode="majority"),
        )

    history.append({
        "pass": 1,
        "stage": "initial_all_topk",
        "report": report_1,
        "verify": verify_1,
    })

    # No verifier or no confession -> stop early
    if (not use_lora) and (not confession):
        return {
            "final_report": report_1,
            "history": history,
            "confession_enabled": bool(confession),
            "passes_run": len(history),
            "majority_flag": majority_flag,
        }

    if use_lora and not confession:
        return {
            "final_report": report_1,
            "history": history,
            "confession_enabled": bool(confession),
            "passes_run": len(history),
            "majority_flag": majority_flag,
        }

    # If confession is ON, always run pass 2 with majority evidence
    messages = build_majority_messages(majority_neighbors, report_1, pass_id=2)
    report_2 = generate_report_base(
        model=model,
        tokenizer=tokenizer,
        prompt_messages=messages,
    )
    verify_2 = verify_true_false_lora(
        model=model,
        tokenizer=tokenizer,
        verify_prompt=build_verify_prompt(report_2, neighbors, mode="majority"),
    ) if use_lora else None

    history.append({
        "pass": 2,
        "stage": "majority_revision",
        "report": report_2,
        "verify": verify_2,
    })

    if STOP_IF_UNCHANGED and normalize_report_text(report_2) == normalize_report_text(report_1):
        return {
            "final_report": report_2,
            "history": history,
            "confession_enabled": True,
            "passes_run": len(history),
            "majority_flag": majority_flag,
        }

    # Case 1: verify_1 TRUE -> end after pass 2
    if use_lora and verify_1 is not None and verify_1.get("pred") == "TRUE":
        return {
            "final_report": report_2,
            "history": history,
            "confession_enabled": True,
            "passes_run": len(history),
            "majority_flag": majority_flag,
        }

    # Case 2: verify_1 FALSE -> inspect pass 2 verifier
    if use_lora and verify_2 is not None and verify_2.get("pred") == "TRUE":
        return {
            "final_report": report_2,
            "history": history,
            "confession_enabled": True,
            "passes_run": len(history),
            "majority_flag": majority_flag,
        }

    # Pass 3: use minority text annotated as false
    messages = build_minority_false_messages(minority_neighbors, report_2, pass_id=3)
    report_3 = generate_report_base(
        model=model,
        tokenizer=tokenizer,
        prompt_messages=messages,
    )
    verify_3 = verify_true_false_lora(
        model=model,
        tokenizer=tokenizer,
        verify_prompt=build_verify_prompt(report_3, neighbors, mode="minority_false"),
    ) if use_lora else None

    history.append({
        "pass": 3,
        "stage": "minority_false_revision",
        "report": report_3,
        "verify": verify_3,
    })

    if use_lora and verify_3 is not None and verify_3.get("pred") == "TRUE":
        return {
            "final_report": report_3,
            "history": history,
            "confession_enabled": True,
            "passes_run": len(history),
            "majority_flag": majority_flag,
        }

    # Pass 4: final mixed pass, always keep as final answer
    messages = build_mixed_final_messages(majority_neighbors, minority_neighbors, report_3, pass_id=4)
    report_4 = generate_report_base(
        model=model,
        tokenizer=tokenizer,
        prompt_messages=messages,
    )
    verify_4 = verify_true_false_lora(
        model=model,
        tokenizer=tokenizer,
        verify_prompt=build_verify_prompt(report_4, neighbors, mode="mixed"),
    ) if use_lora else None

    history.append({
        "pass": 4,
        "stage": "final_mixed_revision",
        "report": report_4,
        "verify": verify_4,
    })

    return {
        "final_report": report_4,
        "history": history,
        "confession_enabled": True,
        "passes_run": len(history),
        "majority_flag": majority_flag,
    }

# ============================================================
# 8C) Rule-based report builder — no LLM, purely deterministic
# ============================================================

# Per-label clinical finding sentences used in the FINDINGS section.
# Each maps a CheXpert label to a short, clinically plausible sentence.
_LABEL_FINDINGS_SENTENCES: Dict[str, str] = {
    "atelectasis":               "Atelectasis is present.",
    "cardiomegaly":              "Cardiomegaly is identified, with increased cardiac silhouette.",
    "consolidation":             "Airspace consolidation is noted in the lung parenchyma.",
    "edema":                     "Pulmonary edema is present with vascular congestion.",
    "enlarged cardiomediastinum":"The cardiomediastinum is widened.",
    "fracture":                  "A fracture is identified.",
    "lung lesion":               "A focal lung lesion is present.",
    "lung opacity":              "Lung opacity is noted, possibly representing atelectasis or infiltrate.",
    "pleural effusion":          "Pleural effusion is present.",
    "pleural other":             "Additional pleural abnormality is noted.",
    "pneumonia":                 "Airspace opacity consistent with pneumonia is identified.",
    "pneumothorax":              "Pneumothorax is present.",
    "support devices":           "Support devices are in place.",
}

# Ordering used to assemble the IMPRESSION line: more urgent conditions first.
_IMPRESSION_PRIORITY = [
    "pneumothorax",
    "pneumonia",
    "consolidation",
    "edema",
    "lung lesion",
    "lung opacity",
    "atelectasis",
    "pleural effusion",
    "pleural other",
    "cardiomegaly",
    "enlarged cardiomediastinum",
    "fracture",
    "support devices",
]

def build_rule_based_report(
    predicted_labels: List[str],
    predicted_normal: str,
) -> str:
    """
    Construct a clinical FINDINGS + IMPRESSION text from majority-voted labels.

    Rules:
    - Each predicted label contributes one sentence to FINDINGS drawn from
      _LABEL_FINDINGS_SENTENCES (no LLM, no randomness).
    - IMPRESSION is assembled in clinical priority order (urgent first).
    - If no labels are predicted the report states a normal examination.

    The label names are preserved verbatim in the output text so that
    extract_labels_from_text() can recover them for metric computation.
    """
    if not predicted_labels:
        findings   = "No acute cardiopulmonary abnormality identified."
        impression = "Normal chest radiograph. No significant findings."
        return f"FINDINGS: {findings}\nIMPRESSION: {impression}"

    # --- FINDINGS: one sentence per predicted label in LABELS_13 order ---
    finding_sentences = []
    for lab in LABELS_13:           # preserve canonical label order
        if lab in predicted_labels:
            sentence = _LABEL_FINDINGS_SENTENCES.get(lab, f"{lab.capitalize()} is present.")
            finding_sentences.append(sentence)
    findings = " ".join(finding_sentences)

    # --- IMPRESSION: list in priority order, then normal/abnormal statement ---
    impression_labels = [
        lab for lab in _IMPRESSION_PRIORITY if lab in predicted_labels
    ]
    # Any labels not covered by priority list (future-proofing)
    impression_labels += [
        lab for lab in predicted_labels if lab not in impression_labels
    ]

    if len(impression_labels) == 1:
        imp_text = impression_labels[0]
    elif len(impression_labels) == 2:
        imp_text = f"{impression_labels[0]} and {impression_labels[1]}"
    else:
        imp_text = ", ".join(impression_labels[:-1]) + f", and {impression_labels[-1]}"

    impression = f"Abnormal chest radiograph. Findings consistent with {imp_text}."

    return f"FINDINGS: {findings}\nIMPRESSION: {impression}"

# ============================================================
# 8B) Pure FAISS majority-vote baseline (no LLM)
# ============================================================

def faiss_majority_vote_predict(
    neighbors: List[Dict[str, Any]],
    k: int = 10,
) -> Dict[str, Any]:
    """
    Pure FAISS majority-vote baseline — no LLM call at all.

    For each of the 13 CheXpert pathology labels and the normal/abnormal
    status, takes a majority vote across the top-k retrieved neighbors:
    - A label is predicted PRESENT  when strictly more than half of the
      top-k neighbors carry that label in their metadata `labels` field.
    - Normal is predicted YES  when strictly more than half carry normal='yes'.

    The predicted labels are embedded into a structured FINDINGS /
    IMPRESSION text so that downstream extract_labels_from_text() and
    compute_full_metrics() work unchanged (identical record schema to
    the LLM-based methods).
    """
    top_n = neighbors[: max(1, min(int(k), len(neighbors)))]
    n = len(top_n)
    threshold = n / 2.0  # strictly more than half

    # Per-label vote counts + normal-yes count
    label_votes: Dict[str, int] = {lab: 0 for lab in LABELS_13}
    normal_yes_count = 0

    for row in top_n:
        row_labs = set(_row_labels(row))
        for lab in LABELS_13:
            if lab in row_labs:
                label_votes[lab] += 1
        if _row_normal(row) == "yes":
            normal_yes_count += 1

    predicted_labels = [lab for lab in LABELS_13 if label_votes[lab] > threshold]
    predicted_normal = "yes" if normal_yes_count > threshold else "no"

    # Build a proper rule-based clinical report (no LLM).
    # build_rule_based_report() produces per-label FINDINGS sentences and
    # a priority-ordered IMPRESSION — see section 8C below for the logic.
    pred_report = build_rule_based_report(predicted_labels, predicted_normal)

    return {
        "final_report": pred_report,
        "history": [
            {
                "pass": 1,
                "stage": "faiss_majority_vote",
                "report": pred_report,
                "verify": None,
                "label_votes": label_votes,
                "normal_yes_votes": normal_yes_count,
                "n_neighbors_used": n,
                "threshold": threshold,
            }
        ],
        "confession_enabled": False,
        "passes_run": 1,
        "majority_flag": predicted_normal,
    }


In [ ]:
# ============================================================
# 8A) Optional sanity check: base generation should NOT return TRUE/FALSE
# ============================================================
# Uncomment for debugging after model load:
# test_neighbors = [{"normal":"yes","labels":[],"caption":"Normal chest radiograph.","FINDINGS":"No focal airspace disease.","IMPRESSION":"No acute cardiopulmonary process."}]
# test_messages = build_initial_messages(test_neighbors, k=1)
# test_report = generate_report_base(
#     model=verifier_base,
#     tokenizer=tokenizer,
#     prompt_messages=test_messages,
# )
# print("Sanity generation:", test_report)


In [10]:
# ============================================================
# 9) Batch prediction over one FAISS test set
#    NOTE: image_path is used ONLY for BioMedCLIP FAISS retrieval.
#    The LLM receives only the retrieved text reports.
#    Works for all 4 experiment methods via use_lora / confession flags.
# ============================================================

def batch_predict_biollama(
    *,
    test_index: faiss.Index,
    test_meta: List[Dict[str, Any]],
    gal_index: faiss.Index,
    gal_meta: List[Dict[str, Any]],
    use_lora: bool,
    confession: bool,
    use_faiss_majority: bool = False,
    n_items: Optional[int] = None,
    k: int = 10,
) -> List[Dict[str, Any]]:
    total = len(test_meta) if n_items is None else min(int(n_items), len(test_meta))
    outputs: List[Dict[str, Any]] = []

    for tid in range(total):
        row = test_meta[tid]
        qvec = query_vec_from_test_id(test_index, tid)
        neighbors = retrieve_gallery_neighbors(gal_index, gal_meta, qvec, k=k)

        if use_faiss_majority:
            # Pure FAISS majority-vote baseline — no LLM call
            pred = faiss_majority_vote_predict(neighbors=neighbors, k=k)
        else:
            pred = rag_generate_biollama_report(
                model=verifier_base,
                tokenizer=tokenizer,
                neighbors=neighbors,
                k=k,
                use_lora=use_lora,
                confession=confession,
            )

        out = {
            "test_id": tid,
            "faiss_id": row.get("faiss_id", tid),
            "image_path": row.get("image_path"),
            "gt_report": build_gt_report_text(row),
            "pred_report": pred["final_report"],
            "history": pred["history"],
            "confession_enabled": pred["confession_enabled"],
            "passes_run": pred["passes_run"],
            "majority_flag": pred.get("majority_flag"),
            "neighbors": [
                {
                    "rank": n.get("_rank"),
                    "score": n.get("_score"),
                    "normal": _row_normal(n),
                    "labels": _row_labels(n),
                    "caption": _row_caption(n),
                    "FINDINGS": _row_findings(n),
                    "IMPRESSION": _row_impression(n),
                }
                for n in neighbors
            ],
        }
        outputs.append(out)

        if (tid + 1) % int(PROGRESS_EVERY) == 0:
            print(f"Processed {tid + 1}/{total}")

    return outputs

In [11]:
# ============================================================
# 10) Full evaluation metrics
# ============================================================
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score
from sklearn.metrics import f1_score

try:
    from pycocoevalcap.cider.cider import Cider
    _HAS_CIDER = True
except Exception:
    _HAS_CIDER = False

LABELS_13 = [
    "atelectasis",
    "cardiomegaly",
    "consolidation",
    "edema",
    "enlarged cardiomediastinum",
    "fracture",
    "lung lesion",
    "lung opacity",
    "pleural effusion",
    "pleural other",
    "pneumonia",
    "pneumothorax",
    "support devices",
]

def _clean_text(x: Any) -> str:
    x = "" if x is None else str(x)
    return re.sub(r"\s+", " ", x).strip()

def extract_labels_from_text(text: str) -> Dict[str, Any]:
    text_low = _clean_text(text).lower()
    labels = []
    for lab in LABELS_13:
        if re.search(rf"\b{re.escape(lab)}\b", text_low):
            labels.append(lab)
    normal = "yes" if len(labels) == 0 else "no"
    return {"labels": labels, "normal": normal}

def build_label_vectors(records: List[Dict[str, Any]]):
    y_true, y_pred, normal_true, normal_pred = [], [], [], []
    for r in records:
        gt = extract_labels_from_text(r.get("gt_report", ""))
        pr = extract_labels_from_text(r.get("pred_report", ""))
        y_true.append([1 if l in gt["labels"] else 0 for l in LABELS_13])
        y_pred.append([1 if l in pr["labels"] else 0 for l in LABELS_13])
        normal_true.append(1 if gt["normal"] == "yes" else 0)
        normal_pred.append(1 if pr["normal"] == "yes" else 0)
    return (
        np.array(y_true, dtype=int),
        np.array(y_pred, dtype=int),
        np.array(normal_true, dtype=int),
        np.array(normal_pred, dtype=int),
    )

def compute_generation_accuracy(records: List[Dict[str, Any]]) -> Dict[str, float]:
    y_true, y_pred, normal_true, normal_pred = build_label_vectors(records)

    if len(records) == 0:
        return {
            "N_text_eval": 0,
            "MajorityNormalAcc": float("nan"),
            "NormalAccuracy": float("nan"),
            "MacroF1": float("nan"),
            "MicroF1": float("nan"),
            "HammingAcc": float("nan"),
        }

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    hamming_acc = float((y_true == y_pred).mean())
    normal_acc = float((normal_true == normal_pred).mean())
    majority_label = 1 if normal_true.mean() >= 0.5 else 0
    majority_normal_acc = float((normal_pred == majority_label).mean())

    return {
        "N_text_eval": int(len(records)),
        "MajorityNormalAcc": majority_normal_acc,
        "NormalAccuracy": normal_acc,
        "MacroF1": float(macro_f1),
        "MicroF1": float(micro_f1),
        "HammingAcc": hamming_acc,
    }

def compute_fer(records: List[Dict[str, Any]]) -> float:
    total_pred = 0
    false_pred = 0
    for r in records:
        gt = extract_labels_from_text(r.get("gt_report", ""))
        pr = extract_labels_from_text(r.get("pred_report", ""))
        gt_set = set(gt["labels"])
        for lab in pr["labels"]:
            total_pred += 1
            if lab not in gt_set:
                false_pred += 1
    return false_pred / max(1, total_pred)

def compute_fer_no_normal(records: List[Dict[str, Any]]) -> float:
    total_pred = 0
    false_pred = 0
    for r in records:
        gt = extract_labels_from_text(r.get("gt_report", ""))
        pr = extract_labels_from_text(r.get("pred_report", ""))
        if gt["normal"] == "yes":
            continue
        gt_set = set(gt["labels"])
        for lab in pr["labels"]:
            total_pred += 1
            if lab not in gt_set:
                false_pred += 1
    return false_pred / max(1, total_pred)

def compute_omission_rate(records: List[Dict[str, Any]]) -> float:
    total_gt = 0
    missed = 0
    for r in records:
        gt = extract_labels_from_text(r.get("gt_report", ""))
        pr = extract_labels_from_text(r.get("pred_report", ""))
        pr_set = set(pr["labels"])
        for lab in gt["labels"]:
            total_gt += 1
            if lab not in pr_set:
                missed += 1
    return missed / max(1, total_gt)

def compute_coverage(records: List[Dict[str, Any]]) -> float:
    vals = []
    for r in records:
        gt = extract_labels_from_text(r.get("gt_report", ""))
        pr = extract_labels_from_text(r.get("pred_report", ""))
        if len(gt["labels"]) == 0:
            continue
        hit = sum(1 for lab in gt["labels"] if lab in pr["labels"])
        vals.append(hit / len(gt["labels"]))
    return float(np.mean(vals)) if vals else 0.0

def compute_bleu_mean(records: List[Dict[str, Any]]) -> float:
    smoothie = SmoothingFunction().method1
    vals = []
    for r in records:
        ref = _clean_text(r.get("gt_report", ""))
        hyp = _clean_text(r.get("pred_report", ""))
        if not ref:
            continue
        vals.append(sentence_bleu([ref.split()], hyp.split() if hyp else [], smoothing_function=smoothie))
    return float(np.mean(vals)) if vals else float("nan")

def compute_rouge_scores(records: List[Dict[str, Any]]) -> Dict[str, float]:
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rl = [], [], []
    for r in records:
        ref = _clean_text(r.get("gt_report", ""))
        hyp = _clean_text(r.get("pred_report", ""))
        if not ref:
            continue
        s = scorer.score(ref, hyp)
        r1.append(s["rouge1"].fmeasure)
        r2.append(s["rouge2"].fmeasure)
        rl.append(s["rougeL"].fmeasure)
    return {
        "ROUGE1_F": float(np.mean(r1)) if r1 else float("nan"),
        "ROUGE2_F": float(np.mean(r2)) if r2 else float("nan"),
        "ROUGEL_F": float(np.mean(rl)) if rl else float("nan"),
    }

def compute_bertscore(records: List[Dict[str, Any]]) -> Dict[str, float]:
    refs, hyps = [], []
    for r in records:
        ref = _clean_text(r.get("gt_report", ""))
        hyp = _clean_text(r.get("pred_report", ""))
        if ref:
            refs.append(ref)
            hyps.append(hyp)
    if not refs:
        return {"BERTScore_P": float("nan"), "BERTScore_R": float("nan"), "BERTScore_F1": float("nan")}
    P, R, F1 = bertscore_score(hyps, refs, lang="en", verbose=False)
    return {
        "BERTScore_P": float(P.mean().item()),
        "BERTScore_R": float(R.mean().item()),
        "BERTScore_F1": float(F1.mean().item()),
    }

def compute_cider(records: List[Dict[str, Any]]) -> float:
    if not _HAS_CIDER:
        return float("nan")
    gts, res = {}, {}
    kept = 0
    for r in records:
        ref = _clean_text(r.get("gt_report", ""))
        hyp = _clean_text(r.get("pred_report", ""))
        if not ref:
            continue
        gts[kept] = [ref]
        res[kept] = [hyp]
        kept += 1
    if kept == 0:
        return float("nan")
    scorer = Cider()
    score, _ = scorer.compute_score(gts, res)
    return float(score)

def compute_radgraph_f1(records: List[Dict[str, Any]], model_type: str = "radgraph-xl") -> Dict[str, float]:
    from radgraph import F1RadGraph

    refs = [str(r.get("gt_report", "")).strip() for r in records]
    hyps = [str(r.get("pred_report", "")).strip() for r in records]
    pairs = [(ref, hyp) for ref, hyp in zip(refs, hyps) if ref and hyp]
    if not pairs:
        return {"RadGraph_E": float("nan"), "RadGraph_ER": float("nan"), "RadGraph_BER": float("nan")}

    refs = [p[0] for p in pairs]
    hyps = [p[1] for p in pairs]

    scorer = F1RadGraph(reward_level="all", model_type=model_type)
    mean_reward, reward_list, hypothesis_annotation_lists, reference_annotation_lists = scorer(hyps=hyps, refs=refs)
    rg_e, rg_er, rg_bar_er = mean_reward
    return {
        "RadGraph_E": float(rg_e),
        "RadGraph_ER": float(rg_er),
        "RadGraph_BER": float(rg_bar_er),
    }

def compute_full_metrics(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    metrics = {}
    metrics.update(compute_generation_accuracy(records))
    metrics["FER"] = compute_fer(records)
    metrics["FER_no_normal"] = compute_fer_no_normal(records)
    metrics["OmissionRate"] = compute_omission_rate(records)
    metrics["Coverage"] = compute_coverage(records)

    metrics["BLEU_mean"] = compute_bleu_mean(records)
    metrics.update(compute_rouge_scores(records))
    metrics.update(compute_bertscore(records))
    metrics["CIDEr"] = compute_cider(records)

    num_records = len(records)
    num_empty = sum(int(not _clean_text(r.get("pred_report", ""))) for r in records)
    avg_pred_len = sum(len(_clean_text(r.get("pred_report", ""))) for r in records) / max(1, num_records)

    metrics["EmptyPredictionRate"] = num_empty / max(1, num_records)
    metrics["AvgPredictionLength"] = float(avg_pred_len)

    try:
        rg = compute_radgraph_f1(records, model_type="radgraph-xl")
        metrics.update(rg)
        metrics["RadGraphF1"] = metrics["RadGraph_ER"]
        metrics["RadGraph_available"] = True
    except Exception as e:
        metrics["RadGraph_available"] = False
        metrics["RadGraph_error"] = repr(e)

    return metrics

In [12]:
# ============================================================
# 11) Helper: rebuild combined comparison CSV from saved outputs
# ============================================================

def load_jsonl_records(jsonl_path: Path) -> List[Dict[str, Any]]:
    records = []
    if not jsonl_path.exists():
        return records
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def _exp_jsonl_name(exp_name: str) -> str:
    return f"biomed_llama_{exp_name}_eval_records.jsonl"

def _exp_summary_name(exp_name: str) -> str:
    return f"biomed_llama_{exp_name}_summary.csv"

def attach_run_metadata_to_summary(
    summary: Dict[str, Any],
    dataset_name: str,
    dataset_root: Path,
    exp: Dict[str, Any],
    records: List[Dict[str, Any]],
) -> Dict[str, Any]:
    summary = dict(summary)

    use_lora          = bool(exp["USE_LORA"])
    use_confession    = bool(exp["USE_CONFESSION"])
    use_faiss_majority = bool(exp.get("USE_FAISS_MAJORITY", False))

    summary["Confession"]        = use_confession
    summary["LoRA_enabled"]      = use_lora
    summary["FAISS_MajorityVote"] = use_faiss_majority
    summary["TopK"]         = int(TOP_K)
    summary["Model"]        = BASE_MODEL_ID
    summary["LoRA"]         = str(LORA_PATH) if use_lora else "OFF"
    summary["Dataset"]      = dataset_name
    summary["DatasetRoot"]  = str(dataset_root)
    summary["N_records"]    = len(records)
    summary["EmptyPredictionRate"] = (
        sum(int(not str(r.get("pred_report", "")).strip()) for r in records) / max(1, len(records))
    )
    return summary

def build_comparison_rows_from_payloads(all_results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    comparison_rows = []
    for payload in all_results:
        s = payload["summary"]
        row = {
            "Dataset"     : payload["dataset"],
            "Experiment"  : payload["experiment"],
            "SaveDir"     : payload["save_dir"],
            "USE_LORA"          : s.get("LoRA_enabled"),
            "USE_CONFESSION"    : s.get("Confession"),
            "FAISS_MajorityVote": s.get("FAISS_MajorityVote"),
            "TopK"        : s.get("TopK"),
            "N_records"   : s.get("N_records"),
            "N_text_eval" : s.get("N_text_eval"),
            "BLEU_mean"   : s.get("BLEU_mean"),
            "ROUGE1_F"    : s.get("ROUGE1_F"),
            "ROUGE2_F"    : s.get("ROUGE2_F"),
            "ROUGEL_F"    : s.get("ROUGEL_F"),
            "BERTScore_P" : s.get("BERTScore_P"),
            "BERTScore_R" : s.get("BERTScore_R"),
            "BERTScore_F1": s.get("BERTScore_F1"),
            "CIDEr"       : s.get("CIDEr"),
            "Coverage"    : s.get("Coverage"),
            "EmptyPredictionRate" : s.get("EmptyPredictionRate"),
            "AvgPredictionLength" : s.get("AvgPredictionLength"),
            "RuntimeSec_total"    : s.get("RuntimeSec_total"),
            "RuntimeSec_per_study": s.get("RuntimeSec_per_study"),
        }
        for key in [
            "MajorityNormalAcc", "NormalAccuracy", "MacroF1", "MicroF1", "HammingAcc",
            "FER", "FER_no_normal", "OmissionRate",
            "RadGraph_E", "RadGraph_ER", "RadGraph_BER", "RadGraphF1", "RadGraph_available"
        ]:
            if key in s:
                row[key] = s.get(key)
        comparison_rows.append(row)
    return comparison_rows

def save_comparison_csv(comparison_rows: List[Dict[str, Any]], out_csv: Path):
    if not comparison_rows:
        print("⚠️ No comparison rows to save.")
        return
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(comparison_rows[0].keys()))
        writer.writeheader()
        for row in comparison_rows:
            writer.writerow(row)
    print("✅ Saved combined comparison CSV:", out_csv)

def rebuild_all_results_from_saved_outputs(
    dataset_roots: Dict[str, Path],
    experiments: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    rebuilt_results = []

    for dataset_name, dataset_root in dataset_roots.items():
        for exp in experiments:
            subdir       = exp["subdir"]
            exp_name     = exp["name"]
            save_dir     = dataset_root / subdir
            result_jsonl = save_dir / _exp_jsonl_name(exp_name)
            summary_csv  = save_dir / _exp_summary_name(exp_name)

            if not result_jsonl.exists():
                print(f"⚠️ Missing JSONL, skipping: {result_jsonl}")
                continue

            records = load_jsonl_records(result_jsonl)
            summary = compute_full_metrics(records)
            summary = attach_run_metadata_to_summary(summary, dataset_name, dataset_root, exp, records)

            with open(summary_csv, "w", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow(["metric", "value"])
                for k, v in summary.items():
                    w.writerow([k, v])

            print("✅ Refreshed summary:", summary_csv)

            rebuilt_results.append({
                "dataset"     : dataset_name,
                "experiment"  : exp_name,
                "save_dir"    : str(save_dir),
                "result_jsonl": str(result_jsonl),
                "summary_csv" : str(summary_csv),
                "summary"     : summary,
            })

    return rebuilt_results


In [13]:
# ============================================================
# 12) Run the 15 evaluation combinations + save JSONL/CSV
#     3 datasets × 5 methods = 15 combinations
# ============================================================

def save_records_and_summary(
    records: List[Dict[str, Any]],
    summary: Dict[str, Any],
    save_dir: Path,
    exp_name: str,
):
    save_dir.mkdir(parents=True, exist_ok=True)

    result_jsonl = save_dir / _exp_jsonl_name(exp_name)
    summary_csv  = save_dir / _exp_summary_name(exp_name)

    with open(result_jsonl, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("✅ Saved records:", result_jsonl)

    with open(summary_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["metric", "value"])
        for k, v in summary.items():
            w.writerow([k, v])
    print("✅ Saved summary:", summary_csv)

    return result_jsonl, summary_csv

def compute_summary(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    if "compute_full_metrics" in globals():
        return compute_full_metrics(records)
    raise RuntimeError("compute_full_metrics must be defined for this notebook.")

def run_one_experiment(dataset_name: str, dataset_root: Path, exp: Dict[str, Any]) -> Dict[str, Any]:
    use_lora           = bool(exp["USE_LORA"])
    use_confession     = bool(exp["USE_CONFESSION"])
    use_faiss_majority = bool(exp.get("USE_FAISS_MAJORITY", False))
    exp_name           = exp["name"]
    subdir             = exp["subdir"]

    test_index, test_meta, gal_index, gal_meta = load_dataset_bundles(dataset_root)
    save_dir = dataset_root / subdir
    save_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print("DATASET          :", dataset_name)
    print("TEST ROOT        :", dataset_root)
    print("SAVE DIR         :", save_dir)
    print("USE_LORA         :", use_lora)
    print("CONFESSION       :", use_confession)
    print("FAISS_MAJORITY   :", use_faiss_majority)
    print("=" * 80)

    start_time = time.time()
    records = batch_predict_biollama(
        test_index=test_index,
        test_meta=test_meta,
        gal_index=gal_index,
        gal_meta=gal_meta,
        use_lora=use_lora,
        confession=use_confession,
        use_faiss_majority=use_faiss_majority,
        n_items=N_ITEMS,
        k=TOP_K,
    )
    elapsed = time.time() - start_time

    summary = compute_summary(records)
    summary["RuntimeSec_total"]    = float(elapsed)
    summary["RuntimeSec_per_study"] = float(elapsed / max(1, len(records)))
    summary["Confession"]            = use_confession
    summary["LoRA_enabled"]          = use_lora
    summary["FAISS_MajorityVote"]    = use_faiss_majority
    summary["TopK"]                 = int(TOP_K)
    summary["Model"]                = BASE_MODEL_ID
    summary["LoRA"]                 = str(LORA_PATH) if use_lora else "OFF"
    summary["Dataset"]              = dataset_name
    summary["DatasetRoot"]          = str(dataset_root)
    summary["N_records"]            = len(records)
    summary["EmptyPredictionRate"]  = (
        sum(int(not str(r.get("pred_report", "")).strip()) for r in records)
        / max(1, len(records))
    )

    result_jsonl, summary_csv = save_records_and_summary(records, summary, save_dir, exp_name)

    return {
        "dataset"     : dataset_name,
        "experiment"  : exp_name,
        "save_dir"    : str(save_dir),
        "result_jsonl": str(result_jsonl),
        "summary_csv" : str(summary_csv),
        "summary"     : summary,
    }

ALL_RESULTS: List[Dict[str, Any]] = []

total_combos = len(DATASET_ROOTS) * len(EXPERIMENTS)
combo_idx    = 0
for dataset_name, dataset_root in DATASET_ROOTS.items():
    for exp in EXPERIMENTS:
        combo_idx += 1
        print(f"\n[{combo_idx}/{total_combos}] dataset={dataset_name}  method={exp['name']}")
        run_payload = run_one_experiment(dataset_name, dataset_root, exp)
        ALL_RESULTS.append(run_payload)

comparison_rows = build_comparison_rows_from_payloads(ALL_RESULTS)
save_comparison_csv(comparison_rows, COMPARISON_CSV)

print("\n================ 15-COMBO RUN SUMMARY ================")
for payload in ALL_RESULTS:
    print(payload["dataset"], "|", payload["experiment"], "|", payload["summary_csv"])



[1/3] dataset=mimic  method=faiss_majority_vote
✅ Loaded dataset bundles for faiss_val_mimic_biomedclip
test size: 634 gallery size: 634

DATASET          : mimic
TEST ROOT        : /data/liangz2/openi/faiss_val_mimic_biomedclip
SAVE DIR         : /data/liangz2/openi/faiss_val_mimic_biomedclip/faiss_majority_vote
USE_LORA         : False
CONFESSION       : False
FAISS_MAJORITY   : True
Processed 25/634
Processed 50/634
Processed 75/634
Processed 100/634
Processed 125/634
Processed 150/634
Processed 175/634
Processed 200/634
Processed 225/634
Processed 250/634
Processed 275/634
Processed 300/634
Processed 325/634
Processed 350/634
Processed 375/634
Processed 400/634
Processed 425/634
Processed 450/634
Processed 475/634
Processed 500/634
Processed 525/634
Processed 550/634
Processed 575/634
Processed 600/634
Processed 625/634


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda:0
✅ Saved records: /data/liangz2/openi/faiss_val_mimic_biomedclip/faiss_majority_vote/biomed_llama_faiss_majority_vote_eval_records.jsonl
✅ Saved summary: /data/liangz2/openi/faiss_val_mimic_biomedclip/faiss_majority_vote/biomed_llama_faiss_majority_vote_summary.csv

[2/3] dataset=openi  method=faiss_majority_vote
✅ Loaded dataset bundles for faiss_val_openi_biomedclip
test size: 400 gallery size: 400

DATASET          : openi
TEST ROOT        : /data/liangz2/openi/faiss_val_openi_biomedclip
SAVE DIR         : /data/liangz2/openi/faiss_val_openi_biomedclip/faiss_majority_vote
USE_LORA         : False
CONFESSION       : False
FAISS_MAJORITY   : True
Processed 25/400
Processed 50/400
Processed 75/400
Processed 100/400
Processed 125/400
Processed 150/400
Processed 175/400
Processed 200/400
Processed 225/400
Processed 250/400
Processed 275/400
Processed 300/400
Processed 325/400
Processed 350/400
Processed 375/400
Processed 400/400


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda:0
✅ Saved records: /data/liangz2/openi/faiss_val_openi_biomedclip/faiss_majority_vote/biomed_llama_faiss_majority_vote_eval_records.jsonl
✅ Saved summary: /data/liangz2/openi/faiss_val_openi_biomedclip/faiss_majority_vote/biomed_llama_faiss_majority_vote_summary.csv

[3/3] dataset=combined  method=faiss_majority_vote
✅ Loaded dataset bundles for faiss_val_combined_biomedclip
test size: 1034 gallery size: 1034

DATASET          : combined
TEST ROOT        : /data/liangz2/openi/faiss_val_combined_biomedclip
SAVE DIR         : /data/liangz2/openi/faiss_val_combined_biomedclip/faiss_majority_vote
USE_LORA         : False
CONFESSION       : False
FAISS_MAJORITY   : True
Processed 25/1034
Processed 50/1034
Processed 75/1034
Processed 100/1034
Processed 125/1034
Processed 150/1034
Processed 175/1034
Processed 200/1034
Processed 225/1034
Processed 250/1034
Processed 275/1034
Processed 300/1034
Processed 325/1034
Processed 350/1034
Processed 375/1034
Processed 400/1034
Proces

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda:0
✅ Saved records: /data/liangz2/openi/faiss_val_combined_biomedclip/faiss_majority_vote/biomed_llama_faiss_majority_vote_eval_records.jsonl
✅ Saved summary: /data/liangz2/openi/faiss_val_combined_biomedclip/faiss_majority_vote/biomed_llama_faiss_majority_vote_summary.csv
✅ Saved combined comparison CSV: /data/liangz2/openi/biomed_llama_all_dataset_15combo_comparison.csv

================ 15-COMBO RUN SUMMARY ================
mimic | faiss_majority_vote | /data/liangz2/openi/faiss_val_mimic_biomedclip/faiss_majority_vote/biomed_llama_faiss_majority_vote_summary.csv
openi | faiss_majority_vote | /data/liangz2/openi/faiss_val_openi_biomedclip/faiss_majority_vote/biomed_llama_faiss_majority_vote_summary.csv
combined | faiss_majority_vote | /data/liangz2/openi/faiss_val_combined_biomedclip/faiss_majority_vote/biomed_llama_faiss_majority_vote_summary.csv


In [ ]:
# ============================================================
# 13) Quick inspection of one saved prediction from the last run
# ============================================================
if ALL_RESULTS:
    last_result_jsonl = Path(ALL_RESULTS[-1]["result_jsonl"])
    if last_result_jsonl.exists():
        with open(last_result_jsonl, "r", encoding="utf-8") as f:
            first_line = next((ln for ln in f if ln.strip()), None)
        if first_line:
            ex = json.loads(first_line)
            print("image_path:", ex["image_path"])
            print("\n--- GT REPORT ---\n")
            print(ex["gt_report"])
            print("\n--- FINAL PRED REPORT ---\n")
            print(ex["pred_report"])
            print("\n--- HISTORY ---\n")
            for h in ex["history"]:
                print(f"[pass {h['pass']}] {h.get('stage')}")
                print(h["report"])
                print("verify:", h.get("verify"))
                print()
